# A100 — OP-12 θ firewall screen with a **hard preflight gate**

**Run:** Runtime → Change runtime type → **A100 GPU**, then **Runtime → Run all**.

This notebook will **refuse to run production** unless predeclared acceptance criteria are met. Before any
expensive measurement it runs a cheap preflight that prints:

```
EQUILIBRATION: PASS/FAIL
ESTIMATED TAU_INT: ... sweeps
PLANNED N_EFF: ...
EXPECTED BLOCK COUNT: ...
OPERATOR RESIDUAL: ...
DENSE CROSS-CHECK: PASS/FAIL
PRODUCTION AUTHORIZED: YES/NO
```

If `NO`, it halts and tells you what to raise — **no A100 time is spent on an uncontrolled run**. The criteria
are fixed constants in the `CONFIG` cell (no post-run goalpost changes). Sampler: full-SU(3) Metropolis (ergodic)
\+ microcanonical SU(2)-subgroup **overrelaxation** (decorrelation, hard-gated). θ uses a residual-stopped power
iteration cross-checked against a **dense machine-precision eigensolve** in the sparse regime. FP64.

GPU output is floating-point evidence; the rational certificate remains the separate CPU job.

In [ ]:
# --- setup: detect GPU, install CuPy if needed (Colab CUDA 12.x) ---
import subprocess, sys
try:
    import cupy as cp
    print("CuPy already present:", cp.__version__)
except Exception:
    print("Installing cupy-cuda12x ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=False)
    import cupy as cp
    print("CuPy installed:", cp.__version__)
try:
    nm = cp.cuda.runtime.getDeviceProperties(0)["name"]; nm = nm.decode() if isinstance(nm, bytes) else nm
    print("GPU:", nm, "(prefer A100 for throughput)" if "A100" not in nm else "")
except Exception as e:
    print("No CUDA GPU visible:", e, "-> CPU fallback (slow).")


In [ ]:
# ===== OP-12 theta firewall screen, preflight-gated (single program cell) =====
#!/usr/bin/env python3
# op12_a100_preflight.py
# OP-12 theta firewall screen with a HARD PREFLIGHT GATE in front of production.
# Protocol (predeclared, no post-run goalpost changes):
#   1. operator + overrelaxation hard gates
#   2. cold/hot EQUILIBRATION test  -> t_eq (or ABORT)
#   3. tau_int estimate (Sokal auto-window) on plaquette AND defect fraction
#   4. expected N_eff and block count computed for the sized production
#   5. dense-exact theta cross-check (operator residual)
#   6. print PRODUCTION AUTHORIZED: YES/NO  -> production runs only if YES
# Sampler: full-SU(3) Metropolis (ergodic) + microcanonical SU(2)-subgroup
# overrelaxation (decorrelation, action-preserving, hard-gated).
# Backend: CuPy/GPU if present else NumPy/CPU -- identical code path. FP64.
import os, sys, json, time, math, argparse
import numpy as np
import scipy.sparse as ssp

USE_GPU = False
try:
    import cupy as cp
    import cupyx.scipy.sparse as xsp_gpu
    if cp.cuda.runtime.getDeviceCount() > 0:
        USE_GPU = True
except Exception:
    cp = None
if USE_GPU:
    xp = cp; xsp = xsp_gpu
    _d = cp.cuda.runtime.getDeviceProperties(cp.cuda.runtime.getDevice())
    BACKEND = f"CuPy/GPU [{_d['name'].decode() if isinstance(_d['name'],bytes) else _d['name']}]"
else:
    xp = np; xsp = ssp; BACKEND = "NumPy/CPU"
def to_host(a): return cp.asnumpy(a) if (USE_GPU and isinstance(a, cp.ndarray)) else np.asarray(a)
def asdev(a):   return cp.asarray(a) if USE_GPU else np.asarray(a)
def dag(M):     return xp.conj(xp.swapaxes(M, -1, -2))

# ================= PREDECLARED ACCEPTANCE CRITERIA (edit before running) =======
CRIT = dict(
    TARGET_N_EFF      = 80,      # required effective independent configs
    MIN_BLOCKS        = 30,      # required jackknife/bootstrap blocks
    TAU_RESOLVE_MULT  = 25,      # preflight tail length must be >= this * tau
    EQ_BAND           = 0.004,   # cold/hot plaquette gap defining "equilibrated"
    EQ_MAX_FRAC       = 0.60,    # equilibration must occur within this frac of preflight sweeps
    XCHECK_TOL        = 1e-3,    # power-vs-dense theta agreement (assembly-bug catcher)
    MAX_PROD_SWEEPS   = 200000,  # COST CAP: refuse if sized production exceeds this
)
# ===============================================================================

def cg_solve(matvec, b, tol=1e-10, maxiter=30000, check_every=10):
    x = xp.zeros_like(b); r = b - matvec(x); p = r.copy()
    rs = xp.dot(r, r); bnorm2 = float(xp.dot(b, b)) + 1e-300; tol2 = (tol*tol)*bnorm2
    if float(rs) <= tol2: return x, 0
    for it in range(maxiter):
        Ap = matvec(p); denom = xp.dot(p, Ap); alpha = rs/denom
        x = x + alpha*p; r = r - alpha*Ap; rs_new = xp.dot(r, r)
        if (it % check_every) == 0 and float(rs_new) <= tol2: return x, 0
        p = r + (rs_new/rs)*p; rs = rs_new
    return x, (0 if float(rs) <= tol2 else 1)

def build_lattice(L):
    D = 4; Ns = L**D; Nl = D*Ns
    ORIS = [(mu, nu) for mu in range(D) for nu in range(mu+1, D)]
    coords = np.array([[s % L, (s//L) % L, (s//L**2) % L, (s//L**3) % L] for s in range(Ns)], dtype=np.int64)
    def sidx(x): return ((x[3]*L+x[2])*L+x[1])*L+x[0]
    nbr = np.zeros((Ns, D), dtype=np.int64); pbr = np.zeros((Ns, D), dtype=np.int64)
    for s in range(Ns):
        for mu in range(D):
            xp_ = coords[s].copy(); xp_[mu] = (xp_[mu]+1) % L
            xm_ = coords[s].copy(); xm_[mu] = (xm_[mu]-1) % L
            nbr[s, mu] = sidx(xp_); pbr[s, mu] = sidx(xm_)
    rows, cols, vals = [], [], []
    for s in range(Ns):
        for mu in range(D):
            l = 4*s+mu; rows += [l, l]; cols += [nbr[s, mu], s]; vals += [1.0, -1.0]
    d0 = ssp.csr_matrix((vals, (rows, cols)), shape=(Nl, Ns))
    rows, cols, vals = [], [], []; Np_ = 6*Ns; plinks = np.zeros((Np_, 4), dtype=np.int64)
    for s in range(Ns):
        for oi, (mu, nu) in enumerate(ORIS):
            p = 6*s+oi; ls = [4*s+mu, 4*nbr[s, mu]+nu, 4*nbr[s, nu]+mu, 4*s+nu]
            plinks[p] = ls; rows += [p]*4; cols += ls; vals += [1.0, 1.0, -1.0, -1.0]
    d1 = ssp.csr_matrix((vals, (rows, cols)), shape=(Np_, Nl))
    assert abs(d1 @ d0).max() == 0.0, 'GATE FAIL: d1 d0 != 0'
    L0 = (d0.T @ d0).tocsr(); L1up = (d1.T @ d1).tocsr()
    return dict(D=D, Ns=Ns, Nl=Nl, ORIS=ORIS, plinks=plinks, nbr=asdev(nbr), pbr=asdev(pbr),
                d0=xsp.csr_matrix(d0), d0T=xsp.csr_matrix(d0.T.tocsr()),
                L0=xsp.csr_matrix(L0), L1up=xsp.csr_matrix(L1up))

# ---- SU(3) ----
_lam = np.zeros((8, 3, 3), dtype=np.complex128)
_lam[0][0,1]=_lam[0][1,0]=1; _lam[1][0,1]=-1j; _lam[1][1,0]=1j; _lam[2][0,0]=1; _lam[2][1,1]=-1
_lam[3][0,2]=_lam[3][2,0]=1; _lam[4][0,2]=-1j; _lam[4][2,0]=1j; _lam[5][1,2]=_lam[5][2,1]=1
_lam[6][1,2]=-1j; _lam[6][2,1]=1j; _lam[7][0,0]=_lam[7][1,1]=1/math.sqrt(3); _lam[7][2,2]=-2/math.sqrt(3)
_LAM_DEV = asdev(_lam); SUBGROUPS = ((0,1),(0,2),(1,2))

def rotation_pool(eps, n):
    a = (xp.random.standard_normal((n, 8), dtype=xp.float64) if USE_GPU
         else np.random.standard_normal((n, 8)).astype(np.float64))
    H = xp.einsum('na,aij->nij', asdev(a).astype(xp.complex128), _LAM_DEV)
    w, V = xp.linalg.eigh(H); R = (V * xp.exp(1j*eps*w)[:, None, :]) @ dag(V)
    return xp.concatenate([R, dag(R)], axis=0)

def haar_field(lat, rng):
    z = rng.standard_normal((lat['Ns'], lat['D'], 3, 3)) + 1j*rng.standard_normal((lat['Ns'], lat['D'], 3, 3))
    q, r = xp.linalg.qr(asdev(z)); d = xp.diagonal(r, axis1=-2, axis2=-1)
    q = q * xp.conj(d/xp.abs(d))[..., None, :]
    det = xp.linalg.det(q); q = q.copy(); q[..., :, 2] = q[..., :, 2]*xp.conj(det)[..., None]
    return q.astype(xp.complex128)

def reunitarize(U):
    c0 = U[..., :, 0]; c1 = U[..., :, 1]
    c0 = c0/xp.linalg.norm(c0, axis=-1, keepdims=True)
    c1 = c1 - xp.sum(xp.conj(c0)*c1, axis=-1, keepdims=True)*c0
    c1 = c1/xp.linalg.norm(c1, axis=-1, keepdims=True)
    return xp.stack([c0, c1, xp.conj(xp.cross(c0, c1))], axis=-1)

def staples(lat, U, mu):
    D, nbr, pbr = lat['D'], lat['nbr'], lat['pbr']
    A = xp.zeros((lat['Ns'], 3, 3), dtype=xp.complex128); smu = nbr[:, mu]
    for nu in range(D):
        if nu == mu: continue
        snu = nbr[:, nu]; A = A + U[smu, nu] @ dag(U[snu, mu]) @ dag(U[:, nu])
        sm = pbr[:, nu]; smmu = nbr[sm, mu]; A = A + dag(U[smmu, nu]) @ dag(U[sm, mu]) @ U[sm, nu]
    return A

def metropolis_sweep(lat, U, beta, pool, pmask, npar):
    Ns = lat['Ns']; npool = pool.shape[0]; acc = xp.zeros((), dtype=xp.int64); att = 0
    for mu in range(lat['D']):
        for color in (0, 1):
            A = staples(lat, U, mu); pm = pmask[color]
            idx = xp.random.randint(0, npool, size=Ns); R = pool[idx]
            Uo = U[:, mu]; Up = R @ Uo
            dS = -(beta/3.0)*xp.real(xp.einsum('sij,sji->s', Up-Uo, A))
            u = xp.random.random(Ns); accept = (dS <= 0) | (u < xp.exp(-xp.clip(dS, 0.0, 700.0)))
            sel = accept & pm; U[:, mu] = xp.where(sel[:, None, None], Up, Uo)
            acc = acc + xp.count_nonzero(sel); att += npar[color]
    return float(to_host(acc))/max(att, 1)

def _su2_or_R(b00, b01, b10, b11):
    q0 = xp.real(b00+b11); q1 = xp.imag(b01+b10); q2 = xp.real(b01-b10); q3 = xp.imag(b00-b11)
    k = xp.maximum(xp.sqrt(q0*q0+q1*q1+q2*q2+q3*q3), 1e-30)
    a0, a1, a2, a3 = q0/k, q1/k, q2/k, q3/k
    vd00, vd01 = a0-1j*a3, -a2-1j*a1; vd10, vd11 = a2-1j*a1, a0+1j*a3
    return (vd00*vd00+vd01*vd10, vd00*vd01+vd01*vd11, vd10*vd00+vd11*vd10, vd10*vd01+vd11*vd11)

def overrelax_sweep(lat, U, pmask):
    eye = xp.broadcast_to(xp.eye(3, dtype=xp.complex128), (lat['Ns'], 3, 3))
    for mu in range(lat['D']):
        for color in (0, 1):
            A = staples(lat, U, mu); sel = pmask[color]          # staple constant across subgroups
            for (i, j) in SUBGROUPS:
                old = U[:, mu]; B = old @ A
                R00, R01, R10, R11 = _su2_or_R(B[:, i, i], B[:, i, j], B[:, j, i], B[:, j, j])
                R = eye.copy(); R[:, i, i] = R00; R[:, i, j] = R01; R[:, j, i] = R10; R[:, j, j] = R11
                U[:, mu] = xp.where(sel[:, None, None], R @ old, old)
    return U

def plaquette_field(lat, U):
    ORIS, nbr = lat['ORIS'], lat['nbr']; out = xp.zeros(6*lat['Ns'], dtype=xp.float64)
    for oi, (mu, nu) in enumerate(ORIS):
        smu = nbr[:, mu]; snu = nbr[:, nu]
        Upl = U[:, mu] @ U[smu, nu] @ dag(U[snu, mu]) @ dag(U[:, nu])
        out[oi::6] = 1.0 - xp.real(xp.einsum('sii->s', Upl))/3.0
    return out

# ---- theta ----
def make_M_matvec(lat, beta, m2):
    L1up = lat['L1up']
    return lambda v: m2*v + (beta/6.0)*(L1up @ v)
def proj_P(lat, f, tol=1e-11):
    b = lat['d0T'] @ f; b = b - b.mean()
    z, info = cg_solve(lambda v: lat['L0'] @ v + 1e-12*v, b, tol=tol)
    assert info == 0, 'GATE FAIL: CG(L0) did not converge'
    z = z - z.mean(); return f - lat['d0'] @ z
def gate_PM(lat, Mmv, rng):
    v = asdev(rng.standard_normal(lat['Nl'])); Pv = proj_P(lat, v); PPv = proj_P(lat, Pv)
    e1 = float(xp.linalg.norm(PPv-Pv))/(1+float(xp.linalg.norm(Pv))); assert e1 < 1e-6, f'P^2!=P {e1:.1e}'
    MPv = Mmv(Pv); PMPv = proj_P(lat, MPv)
    e2 = float(xp.linalg.norm(PMPv-MPv))/(1+float(xp.linalg.norm(MPv))); assert e2 < 1e-5, f'[M,P]!=0 {e2:.1e}'
    return e1, e2
def theta_power(lat, dlinks, Mmv, v0, rng, iters=1000, tol_res=1e-9):
    mask = np.zeros(lat['Nl']); mask[dlinks] = 1.0; mask = asdev(mask)
    def Bmat(x):
        y = proj_P(lat, mask*x); z, info = cg_solve(Mmv, y, tol=1e-10)
        assert info == 0, 'GATE FAIL: CG(M) did not converge'
        return mask*proj_P(lat, z)
    x = asdev(rng.standard_normal(lat['Nl']))*mask; x = x/xp.linalg.norm(x); lam_ = 0.0
    for it in range(iters):
        y = Bmat(x); lam_ = float(xp.dot(x, y)); ny = float(xp.linalg.norm(y))
        if ny == 0.0: return 0.0, 0.0
        resid = float(xp.linalg.norm(y - lam_*x)); x = y/ny
        if it > 5 and resid <= tol_res*max(1.0, abs(lam_)): break
    return v0*lam_, resid/max(1.0, abs(lam_))
def theta_dense_exact(lat, dlinks, Mmv, v0):
    dl = [int(i) for i in dlinks]; nD = len(dl); dl_dev = asdev(np.array(dl, dtype=np.int64))
    Bcols = np.zeros((nD, nD), dtype=np.float64)
    for b, i in enumerate(dl):
        e = xp.zeros(lat['Nl']); e[i] = 1.0; y = proj_P(lat, e)
        z, info = cg_solve(Mmv, y, tol=1e-12); assert info == 0, 'GATE FAIL: dense CG(M)'
        Bcols[:, b] = to_host(proj_P(lat, z)[dl_dev])
    Bsym = 0.5*(Bcols+Bcols.T)
    asym = float(np.max(np.abs(Bcols-Bsym)))/(1+float(np.max(np.abs(Bsym)))); assert asym < 1e-6, f'not self-adjoint {asym:.1e}'
    return v0*float(np.linalg.eigvalsh(Bsym)[-1])
def defect_links(lat, spl, delta):
    bad = spl > delta; nbad = int(to_host(xp.count_nonzero(bad)))
    if nbad == 0: return np.array([], dtype=np.int64), 0
    return np.unique(lat['plinks'][to_host(bad).astype(bool)].ravel()), nbad
def theta_of(lat, spl, delta, Mmv, v0, rng, dense_cap=1500, xcheck=True):
    dlinks, nbad = defect_links(lat, spl, delta); nD = int(len(dlinks))
    if nD == 0: return dict(theta=0.0, nD=0, method='none', xres=0.0, rhoL=0.0)
    res = dict(nD=nD, rhoL=nD/lat['Nl'], xres=0.0)
    if nD <= dense_cap:
        th = theta_dense_exact(lat, dlinks, Mmv, v0); res['method'] = 'dense_exact'
        if xcheck:
            thp, _ = theta_power(lat, dlinks, Mmv, v0, rng); r = abs(th-thp)/max(1.0, abs(th)); res['xres'] = r
            assert r < CRIT['XCHECK_TOL'], f'GATE FAIL: power vs dense theta disagree ({r:.2e})'
        res['theta'] = th
    else:
        th, pr = theta_power(lat, dlinks, Mmv, v0, rng); res['theta'] = th; res['method'] = 'power_only'; res['xres'] = pr
    return res

# ---- gates ----
def gate_overrelax(lat, beta, rng):
    pm = parity_masks(lat)[0]
    U = haar_field(lat, rng); pb = float(1.0 - plaquette_field(lat, U).mean())
    Uor = overrelax_sweep(lat, U.copy(), pm)
    pa = float(1.0 - plaquette_field(lat, Uor).mean())
    move = float(xp.max(xp.abs(Uor - U)))
    uni = float(xp.max(xp.abs(Uor @ dag(Uor) - xp.eye(3, dtype=xp.complex128))))
    assert abs(pa-pb) < 1e-9, f'GATE FAIL: OR not microcanonical dP={abs(pa-pb):.2e}'
    assert move > 1e-3, f'GATE FAIL: OR did not move links {move:.2e}'
    assert uni < 1e-8, f'GATE FAIL: OR broke unitarity {uni:.2e}'
    return abs(pa-pb), move, uni

def parity_masks(lat):
    L = round(lat['Ns']**0.25)
    par = (np.array([[s % L, (s//L) % L, (s//L**2) % L, (s//L**3) % L] for s in range(lat['Ns'])]).sum(1) % 2)
    return ([asdev(par == 0), asdev(par == 1)], [int((par == 0).sum()), int((par == 1).sum())])

# ---- autocorrelation (Sokal automatic window) ----
def tau_int_auto(series, c=6.0):
    x = np.asarray(series, float); n = len(x)
    if n < 16: return float('nan'), 0, float('nan')
    x = x - x.mean(); var = np.dot(x, x)/n
    if var <= 0: return 0.5, 0, 0.0
    tau = 0.5; W = 1
    for w in range(1, n//2):
        rho = np.dot(x[:-w], x[w:])/((n-w)*var); tau += rho; W = w
        if w >= c*tau: break
    return float(tau), int(W), float(tau*math.sqrt((4*W+2)/n))

def _update(lat, U, beta, eps, pool, pm, npar, n_or):
    a = metropolis_sweep(lat, U, beta, pool, pm, npar)
    for _ in range(n_or): U = overrelax_sweep(lat, U, pm)
    return U, a

def _chain(lat, beta, eps, pre, pm, npar, n_or, delta, hot, rng):
    U = haar_field(lat, rng) if hot else (lambda u: (u.__setitem__((slice(None),) , xp.eye(3, dtype=xp.complex128)) or u))(
        xp.zeros((lat['Ns'], lat['D'], 3, 3), dtype=xp.complex128))
    P = []; rhoD = []
    for s in range(pre):
        pool = rotation_pool(eps, 64); U, a = _update(lat, U, beta, eps, pool, pm, npar, n_or)
        if a < 0.35: eps *= 0.94
        elif a > 0.65: eps *= 1.06
        if (s+1) % 20 == 0: U = reunitarize(U)
        spl = plaquette_field(lat, U); P.append(float(1.0 - spl.mean()))
        rhoD.append(float(to_host(xp.count_nonzero(spl > delta)))/spl.shape[0])
    return np.array(P), np.array(rhoD), U, eps

# ============================ PREFLIGHT ============================
def preflight(lat, args, gate_rng):
    print("="*70); print("PREFLIGHT  (predeclared criteria; production runs only if AUTHORIZED)"); print("="*70)
    for k, v in CRIT.items(): print(f"  criterion {k:18s}= {v}")
    pm, npar = parity_masks(lat)
    Mmv = make_M_matvec(lat, args.betas_pf, args.m2)

    # --- gates ---
    e1, e2 = gate_PM(lat, Mmv, gate_rng); dP, mv, uni = gate_overrelax(lat, args.betas_pf, gate_rng)
    print(f"\n[gates] P^2=P={e1:.1e}  [M,P]=0={e2:.1e}  OR_dP={dP:.1e}  OR_move={mv:.2f}  OR_unit={uni:.1e}")

    # --- equilibration: cold vs hot ---
    print(f"[equilibration] cold+hot, {args.pf_sweeps} sweeps each, OR={args.n_or} ...")
    Pc, Dc, Ueq, eps = _chain(lat, args.betas_pf, args.eps0, args.pf_sweeps, pm, npar, args.n_or, args.deltas_pf, False, gate_rng)
    Ph, Dh, _, _ = _chain(lat, args.betas_pf, args.eps0, args.pf_sweeps, pm, npar, args.n_or, args.deltas_pf, True, gate_rng)
    w = 5; sc = np.convolve(Pc, np.ones(w)/w, 'valid'); sh = np.convolve(Ph, np.ones(w)/w, 'valid'); gap = np.abs(sc-sh)
    sigma_eq = float(np.std(sc[max(1, 2*len(sc)//3):])) if len(sc) > 6 else float(np.std(sc))
    band = max(CRIT['EQ_BAND'], 3.0*sigma_eq)   # gap must fall below absolute floor OR 3x the noise floor
    t_eq = None
    for t in range(len(gap)):
        if np.all(gap[t:] < band): t_eq = t + w//2; break
    eq_pass = (t_eq is not None) and (t_eq <= CRIT['EQ_MAX_FRAC']*args.pf_sweeps)
    final_gap = float(gap[-1])

    # --- tau_int on the equilibrated tail (max of plaquette and defect-fraction) ---
    t0 = t_eq if (t_eq is not None) else args.pf_sweeps//2
    tail_p = Pc[t0:]; tail_d = Dc[t0:]
    tau_p, Wp, _ = tau_int_auto(tail_p); tau_d, Wd, _ = tau_int_auto(tail_d)
    taus = [t for t in (tau_p, tau_d) if np.isfinite(t)]
    tau = max(taus) if taus else float('nan')
    tau_resolved = np.isfinite(tau) and (len(tail_p) >= CRIT['TAU_RESOLVE_MULT']*tau)

    # --- size production from tau, then compute expected power ---
    sep = max(1, int(math.ceil(2.0*tau))) if np.isfinite(tau) else 1
    n_meas = max(CRIT['TARGET_N_EFF'], CRIT['MIN_BLOCKS'])
    therm_prod = max(args.pf_sweeps, int(math.ceil(2.0*(t_eq if t_eq else args.pf_sweeps))))
    n_eff = n_meas/max(1.0, (2.0*tau)/sep) if np.isfinite(tau) else float('nan')
    block_size = max(1, int(math.ceil((2.0*tau)/sep))) if np.isfinite(tau) else 1
    block_count = n_meas//block_size
    total_sweeps = therm_prod + n_meas*sep
    cost_ok = total_sweeps <= CRIT['MAX_PROD_SWEEPS']

    # --- operator residual + dense cross-check on an equilibrated config ---
    spl = plaquette_field(lat, Ueq); dlinks, nbad = defect_links(lat, spl, args.deltas_pf); nD = int(len(dlinks))
    if nD == 0:
        op_res, xc_pass, xc_res, th_dump = 0.0, True, 0.0, 0.0
    elif nD <= args.dense_cap:
        th_d = theta_dense_exact(lat, dlinks, Mmv, args.v0); th_p, op_res = theta_power(lat, dlinks, Mmv, args.v0, gate_rng)
        xc_res = abs(th_d-th_p)/max(1.0, abs(th_d)); xc_pass = xc_res < CRIT['XCHECK_TOL']; th_dump = th_d
    else:
        th_p, op_res = theta_power(lat, dlinks, Mmv, args.v0, gate_rng); xc_res = op_res
        xc_pass = True; th_dump = th_p
        print(f"   note: |D|={nD}>dense_cap at delta={args.deltas_pf}; preflight theta is residual-stopped power (res={op_res:.1e})")

    authorized = bool(eq_pass and tau_resolved and (n_eff >= CRIT['TARGET_N_EFF']) and
                      (block_count >= CRIT['MIN_BLOCKS']) and cost_ok and xc_pass)

    print("\n" + "-"*70)
    print(f"EQUILIBRATION: {'PASS' if eq_pass else 'FAIL'}   (t_eq={t_eq} sweeps, final cold/hot gap={final_gap:.4f}, band={band:.4f})")
    print(f"ESTIMATED TAU_INT: {tau:.1f} sweeps   (plaq={tau_p:.1f} W={Wp}, defect={tau_d:.1f} W={Wd}; using max){'' if tau_resolved else '  [UNRESOLVED: tail too short]'}")
    print(f"PLANNED PRODUCTION: therm={therm_prod}, N_meas={n_meas}, sep={sep}")
    print(f"PLANNED N_EFF: {n_eff:.0f}   (target {CRIT['TARGET_N_EFF']})")
    print(f"EXPECTED BLOCK COUNT: {block_count}   (min {CRIT['MIN_BLOCKS']}, block_size={block_size})")
    print(f"OPERATOR RESIDUAL: {op_res:.1e}")
    print(f"DENSE CROSS-CHECK: {'PASS' if xc_pass else 'FAIL'}   (res={xc_res:.1e}, tol={CRIT['XCHECK_TOL']})")
    print(f"TOTAL PRODUCTION SWEEPS: {total_sweeps}   (cap {CRIT['MAX_PROD_SWEEPS']}, {'OK' if cost_ok else 'EXCEEDED'})")
    print(f"PRODUCTION AUTHORIZED: {'YES' if authorized else 'NO'}")
    print("-"*70)
    if not authorized:
        reasons = []
        if not eq_pass: reasons.append("not equilibrated within preflight budget (raise pf_sweeps or eps tuning)")
        if not tau_resolved: reasons.append("tau_int unresolved (raise pf_sweeps)")
        if np.isfinite(n_eff) and n_eff < CRIT['TARGET_N_EFF']: reasons.append("planned N_eff below target")
        if block_count < CRIT['MIN_BLOCKS']: reasons.append("too few blocks")
        if not cost_ok: reasons.append(f"sized production {total_sweeps} > cap {CRIT['MAX_PROD_SWEEPS']} (improve sampler/raise OR or relax scope)")
        if not xc_pass: reasons.append("dense cross-check failed (operator assembly bug)")
        print("REASON(S):", "; ".join(reasons))
    return dict(authorized=authorized, therm=therm_prod, n_meas=n_meas, sep=sep, tau=tau,
                n_eff=n_eff, block_count=block_count, n_or=args.n_or, eps=eps, U0=Ueq)

# ============================ PRODUCTION ============================
def run_production(lat, args, plan, gate_rng):
    pm, npar = parity_masks(lat); betas = [float(b) for b in args.betas.split(',')]
    deltas = [float(d) for d in args.deltas.split(',')]; os.makedirs(args.out, exist_ok=True)
    print("\n" + "="*70); print("PRODUCTION (authorized)"); print("="*70)
    summary = {}
    for beta in betas:
        tag = f"L{args.L}_b{beta}"; Mmv = make_M_matvec(lat, beta, args.m2)
        gate_PM(lat, Mmv, gate_rng)
        U = xp.zeros((lat['Ns'], lat['D'], 3, 3), dtype=xp.complex128); U[:] = xp.eye(3, dtype=xp.complex128); eps = args.eps0
        t0 = time.time()
        for it in range(plan['therm']):
            pool = rotation_pool(eps, 64); U, a = _update(lat, U, beta, eps, pool, pm, npar, plan['n_or'])
            if a < 0.35: eps *= 0.94
            elif a > 0.65: eps *= 1.06
            if (it+1) % 20 == 0: U = reunitarize(U)
        res = {'cfg': []}; t1 = time.time()
        for c in range(plan['n_meas']):
            for _ in range(plan['sep']):
                pool = rotation_pool(eps, 64); U, a = _update(lat, U, beta, eps, pool, pm, npar, plan['n_or'])
                if a < 0.35: eps *= 0.94
                elif a > 0.65: eps *= 1.06
            U = reunitarize(U); spl = plaquette_field(lat, U); mp = float(1.0 - spl.mean())
            assert -0.05 < mp < 1.05, f'plaquette out of range {mp}'
            entry = {'cfg': c, 'plaq': mp}
            for dl in deltas:
                out = theta_of(lat, spl, dl, Mmv, args.v0, gate_rng, dense_cap=args.dense_cap, xcheck=(c < args.xcheck_ncfg))
                entry[f'theta_d{dl}'] = out['theta']; entry[f'method_d{dl}'] = out['method']; entry[f'nD_d{dl}'] = out['nD']
            res['cfg'].append(entry)
        if USE_GPU: cp.cuda.runtime.deviceSynchronize()
        plaqs = [e['plaq'] for e in res['cfg']]; tau = tau_int_auto(plaqs)[0]
        json.dump(res, open(os.path.join(args.out, f'results_{tag}.json'), 'w'), indent=1)
        meta = dict(L=args.L, beta=beta, backend=BACKEND, n_or=plan['n_or'], therm=plan['therm'],
                    n_meas=plan['n_meas'], sep=plan['sep'], mean_plaq=float(np.mean(plaqs)),
                    tau_int_prod=tau, n_eff_prod=len(plaqs)/max(1.0, 2*tau) if np.isfinite(tau) else None,
                    t_therm=t1-t0, t_meas=time.time()-t1)
        for dl in deltas:
            ths = [e[f'theta_d{dl}'] for e in res['cfg']]
            meta[f'theta_max_d{dl}'] = float(max(ths)); meta[f'theta_p50_d{dl}'] = float(np.median(ths))
        json.dump(meta, open(os.path.join(args.out, f'meta_{tag}.json'), 'w'), indent=1); summary[tag] = meta
        line = f"[{tag}] <plaq>={np.mean(plaqs):.4f} tau_prod={tau:.2f} N_eff={meta['n_eff_prod']:.0f}" if meta['n_eff_prod'] else f"[{tag}]"
        for dl in deltas:
            ths = [e[f'theta_d{dl}'] for e in res['cfg']]; m = res['cfg'][0].get(f'method_d{dl}', '?')
            verdict = 'FIREWALL HOLDS' if np.median(ths) < 1 else 'EXCEEDS 1'
            line += f"  | d={dl}[{m}]: max={max(ths):.4f} p50={np.median(ths):.4f} -> {verdict}"
        print(line)
    json.dump(summary, open(os.path.join(args.out, 'summary.json'), 'w'), indent=1)
    _plot(args, summary, deltas); return summary

def _plot(args, summary, deltas):
    try:
        import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    except Exception: return
    tags = [t for t in summary if t.startswith(f'L{args.L}_b')]
    if not tags: return
    bs = sorted({summary[t]['beta'] for t in tags}); plt.figure(figsize=(7, 4.3))
    for dl in deltas:
        plt.plot(bs, [summary[f'L{args.L}_b{b}'].get(f'theta_max_d{dl}', np.nan) for b in bs], marker='o', label=f'theta_max d={dl}')
    plt.axhline(1.0, ls='--', lw=1, color='k', label='firewall edge'); plt.xlabel('beta'); plt.ylabel('theta')
    plt.title(f'OP-12 theta (L={args.L}, preflight-authorized)'); plt.legend(); plt.tight_layout()
    p = os.path.join(args.out, f'theta_vs_beta_L{args.L}.png'); plt.savefig(p, dpi=160)
    try: plt.show()
    except Exception: pass
    print('saved plot:', p)

def main(args):
    print(f"backend: {BACKEND}"); lat = build_lattice(args.L)
    print(f"lattice: L={args.L} sites={lat['Ns']} links={lat['Nl']}")
    if USE_GPU: xp.random.seed(args.seed)
    else: np.random.seed(args.seed)
    gate_rng = np.random.default_rng(args.seed + 99)
    plan = preflight(lat, args, gate_rng)
    if not plan['authorized']:
        print("\nPRODUCTION HALTED -- acceptance criteria not met. No expensive measurement was run.")
        return None
    return run_production(lat, args, plan, gate_rng)

def selftest():
    print("SELF-TEST: preflight gate + production on tiny config (validates gates + abort/authorize paths)")
    ns = argparse.Namespace(L=4, betas='6.0', deltas='0.9', m2=0.5, v0=1.0, seed=20260614,
                            eps0=0.22, n_or=3, pf_sweeps=220, betas_pf=6.0, deltas_pf=0.9,
                            dense_cap=1500, xcheck_ncfg=2, out='op12_pf_selftest')
    # tiny criteria so the tiny run can actually authorize
    CRIT.update(TARGET_N_EFF=8, MIN_BLOCKS=4, MAX_PROD_SWEEPS=100000, TAU_RESOLVE_MULT=8)
    return main(ns)


# ===================== CONFIG: PREDECLARED ACCEPTANCE CRITERIA =====================
# These are the gate. They are fixed here and NOT changed after the run.
CRIT.update(
    TARGET_N_EFF      = 80,       # required effective independent configs
    MIN_BLOCKS        = 30,       # required jackknife/bootstrap blocks
    TAU_RESOLVE_MULT  = 25,       # preflight tail must be >= this * tau_int
    EQ_BAND           = 0.004,    # absolute floor for cold/hot plaquette agreement
    EQ_MAX_FRAC       = 0.60,     # equilibration must occur within this frac of preflight sweeps
    XCHECK_TOL        = 1e-3,     # power-vs-dense theta agreement (assembly-bug catcher)
    MAX_PROD_SWEEPS   = 200000,   # COST CAP: refuse if the sized production exceeds this
)
# ============================= CONFIG: RUN PARAMETERS ==============================
from types import SimpleNamespace
OUT = "/content/op12_pf_out" if os.path.isdir("/content") else "./op12_pf_out"
RUN = SimpleNamespace(
    L           = 8,              # isotropic L^4 lattice
    betas       = "6.0,6.4",      # production beta ladder
    deltas      = "0.9,1.1",      # production defect thresholds
    betas_pf    = 6.0,            # beta used for the preflight (use your hardest/lowest beta)
    deltas_pf   = 0.9,            # delta used for the preflight tau/defect estimate
    pf_sweeps   = 400,            # preflight sweeps per cold/hot chain (raise if it asks)
    n_or        = 4,              # overrelaxation sweeps per Metropolis sweep
    eps0        = 0.22, m2 = 0.5, v0 = 1.0,
    dense_cap   = 1500,           # exact dense eigensolve when |D| <= this
    xcheck_ncfg = 3,              # power-vs-dense cross-check on this many configs/beta
    seed        = 20260614, out = OUT,
)
# ==================================================================================
summary = main(RUN)   # preflight -> (authorized? production + inline plot : HALT)
